# Build transaction draft features for the additive production-value model

This stage maps draft assets and prepares additive draft features at the transaction-row level.

In [1]:
# Set the processed-data location and load analysis libraries.
from pathlib import Path
import numpy as np
import pandas as pd

processed_data_path = Path("../data/processed")

In [2]:
# Load transaction production features and ranked draft assets.
transaction_features = pd.read_parquet(processed_data_path / "transaction_level_features_offseason.parquet")

transaction_draft_assets = pd.read_parquet("../data/interim/transaction_draft_assets_ranked.parquet")

print("Transaction rows:", len(transaction_features))
print("Draft-asset rows:", len(transaction_draft_assets))

Transaction rows: 2749
Draft-asset rows: 4815


## Map draft assets to the transaction-row identifier

In [3]:
# Validate the transaction keys required for a safe merge.
required_transaction_columns = {"transaction_row_id", "index"}

missing_transaction_columns = required_transaction_columns - set(transaction_features.columns)
if missing_transaction_columns:
    raise KeyError(f"Transaction data is missing: {sorted(missing_transaction_columns)}")

required_draft_columns = {"source_transaction_index", "transaction_side", "point_in_time_tier", "pick_count"}

missing_draft_columns = required_draft_columns - set(transaction_draft_assets.columns)
if missing_draft_columns:
    raise KeyError(f"Draft-asset data is missing: {sorted(missing_draft_columns)}")

transaction_row_lookup = transaction_features[["transaction_row_id", "index"]].rename(columns={"index": "source_transaction_index"}).copy()

transaction_row_lookup["source_transaction_index"] = pd.to_numeric(transaction_row_lookup["source_transaction_index"], errors="coerce")

if not transaction_row_lookup["source_transaction_index"].is_unique:
    raise ValueError("source_transaction_index is not unique in transaction data.")

transaction_draft_assets_in_scope = transaction_draft_assets.merge(
    transaction_row_lookup, on="source_transaction_index", how="inner", validate="many_to_one"
)

print("In-scope transactions with draft assets:", transaction_draft_assets_in_scope["transaction_row_id"].nunique())

In-scope transactions with draft assets: 2080


## Normalize transaction side and draft-asset units

In [4]:
# Normalize draft-asset sides and keep in-scope rows.
transaction_draft_assets_in_scope["transaction_side_clean"] = (
    transaction_draft_assets_in_scope["transaction_side"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

valid_sides = {"acquired", "relinquished"}
observed_sides = set(transaction_draft_assets_in_scope["transaction_side_clean"].dropna())

if not observed_sides.issubset(valid_sides):
    raise ValueError(f"Unexpected transaction sides: {sorted(observed_sides - valid_sides)}")

transaction_draft_assets_in_scope["draft_asset_units"] = pd.to_numeric(
    transaction_draft_assets_in_scope["pick_count"], errors="coerce"
).fillna(1.0)

## Aggregate draft counts by transaction, side, and tier

In [5]:
# Create additive acquired and relinquished counts for every draft tier.
draft_tier_values = sorted(transaction_draft_assets["point_in_time_tier"].dropna().astype(str).unique())

draft_side_values = ["acquired", "relinquished"]

transaction_draft_tier_counts = transaction_draft_assets_in_scope.pivot_table(
    index="transaction_row_id",
    columns=["transaction_side_clean", "point_in_time_tier"],
    values="draft_asset_units",
    aggfunc="sum",
    fill_value=0,
)

expected_draft_columns = pd.MultiIndex.from_product(
    [draft_side_values, draft_tier_values], names=["transaction_side_clean", "point_in_time_tier"]
)

transaction_draft_tier_counts = transaction_draft_tier_counts.reindex(columns=expected_draft_columns, fill_value=0)

transaction_draft_tier_counts.columns = [f"{side}_{tier}_count" for side, tier in transaction_draft_tier_counts.columns]

transaction_draft_tier_counts = transaction_draft_tier_counts.reset_index()

transaction_draft_side_counts = (
    transaction_draft_assets_in_scope.pivot_table(
        index="transaction_row_id", columns="transaction_side_clean", values="draft_asset_units", aggfunc="sum", fill_value=0
    )
    .reindex(columns=draft_side_values, fill_value=0)
    .rename(columns={"acquired": "acquired_total_draft_asset_count", "relinquished": "relinquished_total_draft_asset_count"})
    .reset_index()
)

transaction_draft_summary = transaction_draft_tier_counts.merge(
    transaction_draft_side_counts, on="transaction_row_id", how="outer", validate="one_to_one"
)

## Keep transaction context and player-data coverage fields

In [6]:
# Retain transaction identifiers and model features.
identifier_candidates = ["transaction_row_id", "index", "Date", "Team", "Season", "Acquired", "Relinquished", "Notes"]

coverage_candidates = [
    "acquired_listed_player_count",
    "relinquished_listed_player_count",
    "acquired_calculated_player_count",
    "relinquished_calculated_player_count",
    "acquired_player_id_missing_count",
    "relinquished_player_id_missing_count",
    "acquired_unresolved_name_count",
    "relinquished_unresolved_name_count",
    "acquired_no_prior_appearance_count",
    "relinquished_no_prior_appearance_count",
    "acquired_no_prior_nba_appearance_count",
    "relinquished_no_prior_nba_appearance_count",
    "asset_only_transaction",
]

base_columns = [column for column in [*identifier_candidates, *coverage_candidates] if column in transaction_features.columns]

transaction_draft_features = (
    transaction_features[base_columns].copy().merge(transaction_draft_summary, on="transaction_row_id", how="left", validate="one_to_one")
)

draft_count_columns = [
    column
    for column in transaction_draft_features.columns
    if (
        column.startswith(("acquired_", "relinquished_"))
        and column.endswith("_count")
        and (
            column == "acquired_total_draft_asset_count"
            or column == "relinquished_total_draft_asset_count"
            or any(tier in column for tier in draft_tier_values)
        )
    )
]

transaction_draft_features[draft_count_columns] = transaction_draft_features[draft_count_columns].fillna(0.0).astype("float64")

transaction_draft_features["has_any_draft_compensation"] = transaction_draft_features["acquired_total_draft_asset_count"].gt(
    0
) | transaction_draft_features["relinquished_total_draft_asset_count"].gt(0)

## Validate and save

In [7]:
# Validate row counts and save the additive draft-feature table.
if len(transaction_draft_features) != len(transaction_features):
    raise AssertionError("The draft merge changed the transaction row count.")

if not transaction_draft_features["transaction_row_id"].is_unique:
    raise AssertionError("transaction_row_id is not unique after the draft merge.")

if transaction_draft_features.columns.duplicated().any():
    raise AssertionError("Duplicate output columns were created.")

output_path = processed_data_path / "transaction_draft_features_additive.parquet"

transaction_draft_features.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(transaction_draft_features))
print("Transactions with draft compensation:", int(transaction_draft_features["has_any_draft_compensation"].sum()))

Saved: ..\data\processed\transaction_draft_features_additive.parquet
Rows: 2749
Transactions with draft compensation: 2080
